In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 7.4 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://baritone-scarily-unmade.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://baritone-scarily-unmade.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [10]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學 (Ming Hsin University of Science and Technology, 簡稱明新科大) 是一所位於台灣新竹縣的知名私立科技大學。

以下是其主要簡介：

1.  **創立與歷史地位**：
    *   創立於**1966年**，前身為「明新工業專科學校」。
    *   它不僅是**全國第一所由「工業專科學校」成功改制為「科技大學」的私立學校**，也是台灣歷史最悠久、規模最大的私立科技大學之一，深耕技職教育超過半世紀。

2.  **辦學理念與特色**：
    *   **務實致用**：以「誠信、精勤」為校訓，秉持「務實致用」的辦學理念，致力於培養具備專業技能、實務經驗與人文素養的產業人才。
    *   **產業鏈結**：由於地處新竹科學園區旁，與在地高科技產業及服務業的鏈結非常緊密，在產學合作、人才培育與技術交流方面表現卓越。
    *   **高就業率**：畢業生深受企業界肯定，擁有優良的就業率與職場競爭力，許多校友都在各行各業有傑出表現。

3.  **學術架構與學習資源**：
    *   設有工學院、管理學院、服務事業學院、人文社會學院等，提供學士、碩士等多層次的教育。
    *   課程強調實作、實驗與專題研究，並積極輔導學生考取專業證照，增加職場競爭力。
    *   校園環境優美，設有現代化的教學設施、圖書館、體育館及完善的學生宿舍與生活機能。

4.  **國際化與永續發展**：
    *   近年來積極推動國際交流與合作，與多國姊妹校簽訂協議，提供學生交換學習、雙聯學位及海外實習機會，拓展學生的國際視野。
    *   重視永續發展，積極將SDGs目標融入校務發展與教學研究中。

總結來說，明新科技大學是一所歷史悠久、以實務導向聞名，並與產業緊密結合的科技大學，為台灣培育了無數優秀的專業人才。


In [9]:
result2 = stateful_query("校長是誰？")
print(result2)

我不知道您指的是哪一所學校的校長。因為每所學校都有自己的校長。

如果您能告訴我您想知道的是哪一所學校，我或許能為您查詢相關資訊。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"U119d07e95858d5d83008458c2d433c76","events":[{"type":"message","message":{"type":"text","id":"616073675708039630","quoteToken":"90A3Kdo47Eqom_RJ2PQ0L5Tltskbh9hU4FEnSWFYSz0Fq0LGgWOEhoECRba2zuavw81Ur35OINi2pB-hlwJ8yb4fwKf7MjyYA5j1rZIKoWu130ZuBa2MmOQFEmwHOOgTtA3dJ6Aat1Fzw4PUwsF6-w","markAsReadToken":"w0q5dnUl4XP5ARNjW5sghHJm26uk1f-4T31_ELmMVCNAzQlAbUk5K_1FQwPzJdnrKvwO1oRPNF62ksIZA_5FzCTkciOaZzxrdqVY02a5JZxBhVnIEpI-C1VTp0ET1_rHRBhkU8CYDewK4irQjyWLI1VUGwniZ46TOhbyFiLbPUf94tkc_yNQ11XHDKnkyWB49sUPZ__4ZCOJktto5Z0WhQ","text":"AI 簡介明新科技大學"},"webhookEventId":"01KSSA8JZXR6VTH4CHDTTP2K6H","deliveryContext":{"isRedelivery":true},"timestamp":1780039830016,"source":{"type":"user","userId":"U3d0a7956380119affa33de4fb575bbcb"},"replyToken":"6899ad6070534623a9118cb8e152101b","mode":"active"}]}


ERROR:__main__:Exception on / [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1054/1799486110.py", line 39, in callback
    handler.handle(body, signature)
  File "/usr/local/lib/python3.12/d

BODY:  {"destination":"U119d07e95858d5d83008458c2d433c76","events":[{"type":"message","message":{"type":"text","id":"616074319717990609","quoteToken":"kr8Bkx8gl8EJDfNOk3-DozaF1g_I2hh7BmhUqWLMl4gn0Y9OP3KrzNB6w4bLRD2rzDr5VGieAR0sPO3sbSwswr7jcYp-61TgYUTOkWCatdqaqpf9zlKluc751hUgtiCc_-0dMTR2Tn6a8kE6pmsW7Q","markAsReadToken":"ZInFlAaHvzk5tsuW5OWJa2sAoPKkGpY8YDocHRrCb7B2abGl9eQHq5Z9xQOzUHTQE6FE2QIuIEUFPA3uhdZDuLN6aA5GXhLKHPm08K8huw5XbFqfhH9aVtltEFvMEeg50tttkoX5BbrTU9PqMxOz0wf0zAEvr9qQdHfe2BMzZ2RL3fQi0cL8ybGytmEuOXUn0NYVvsKdlla6O5oCIqg0wQ","text":"AI 介紹明新科大，20字之內"},"webhookEventId":"01KSSAM9Y2J0V49092Y7PNT5T4","deliveryContext":{"isRedelivery":false},"timestamp":1780040213957,"source":{"type":"user","userId":"U3d0a7956380119affa33de4fb575bbcb"},"replyToken":"cc631d7cb8d74ee2bf88230db69bd10a","mode":"active"}]}


...

INFO:werkzeug:127.0.0.1 - - [29/May/2026 07:37:11] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U119d07e95858d5d83008458c2d433c76","events":[{"type":"message","message":{"type":"text","id":"616073675708039630","quoteToken":"90A3Kdo47Eqom_RJ2PQ0L5Tltskbh9hU4FEnSWFYSz0Fq0LGgWOEhoECRba2zuavw81Ur35OINi2pB-hlwJ8yb4fwKf7MjyYA5j1rZIKoWu130ZuBa2MmOQFEmwHOOgTtA3dJ6Aat1Fzw4PUwsF6-w","markAsReadToken":"w0q5dnUl4XP5ARNjW5sghHJm26uk1f-4T31_ELmMVCNAzQlAbUk5K_1FQwPzJdnrKvwO1oRPNF62ksIZA_5FzCTkciOaZzxrdqVY02a5JZxBhVnIEpI-C1VTp0ET1_rHRBhkU8CYDewK4irQjyWLI1VUGwniZ46TOhbyFiLbPUf94tkc_yNQ11XHDKnkyWB49sUPZ__4ZCOJktto5Z0WhQ","text":"AI 簡介明新科技大學"},"webhookEventId":"01KSSA8JZXR6VTH4CHDTTP2K6H","deliveryContext":{"isRedelivery":true},"timestamp":1780039830016,"source":{"type":"user","userId":"U3d0a7956380119affa33de4fb575bbcb"},"replyToken":"6899ad6070534623a9118cb8e152101b","mode":"active"}]}


ERROR:__main__:Exception on / [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1054/1799486110.py", line 39, in callback
    handler.handle(body, signature)
  File "/usr/local/lib/python3.12/d

BODY:  {"destination":"U119d07e95858d5d83008458c2d433c76","events":[{"type":"message","message":{"type":"text","id":"616073675708039630","quoteToken":"90A3Kdo47Eqom_RJ2PQ0L5Tltskbh9hU4FEnSWFYSz0Fq0LGgWOEhoECRba2zuavw81Ur35OINi2pB-hlwJ8yb4fwKf7MjyYA5j1rZIKoWu130ZuBa2MmOQFEmwHOOgTtA3dJ6Aat1Fzw4PUwsF6-w","markAsReadToken":"w0q5dnUl4XP5ARNjW5sghHJm26uk1f-4T31_ELmMVCNAzQlAbUk5K_1FQwPzJdnrKvwO1oRPNF62ksIZA_5FzCTkciOaZzxrdqVY02a5JZxBhVnIEpI-C1VTp0ET1_rHRBhkU8CYDewK4irQjyWLI1VUGwniZ46TOhbyFiLbPUf94tkc_yNQ11XHDKnkyWB49sUPZ__4ZCOJktto5Z0WhQ","text":"AI 簡介明新科技大學"},"webhookEventId":"01KSSA8JZXR6VTH4CHDTTP2K6H","deliveryContext":{"isRedelivery":true},"timestamp":1780039830016,"source":{"type":"user","userId":"U3d0a7956380119affa33de4fb575bbcb"},"replyToken":"6899ad6070534623a9118cb8e152101b","mode":"active"}]}


ERROR:__main__:Exception on / [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1054/1799486110.py", line 39, in callback
    handler.handle(body, signature)
  File "/usr/local/lib/python3.12/d